# Project 20 — RAG From Scratch (chunk → embed → index → retrieve → prompt → read)

```
48 synthetic docs (6 topics x 8, unique entity per doc, planted numeric facts)
  -> word chunker (size/overlap sweep) -> TF-IDF + BM25 from scratch (numpy, no libs)
  -> brute-force cosine index -> top-k retrieve -> bigram rerank -> prompt build
  -> extractive reader (offline stand-in for the LLM) vs closed-book baseline
  -> metrics: recall@k, MRR (retrieval) vs EM (generation); split: EM|hit vs EM|miss
  -> artifacts: results/retrieval.csv + eval.csv + prompt_example.txt + rag.png
```

Goal: separate the two things RAG can get wrong — finding the evidence (retrieval) vs using it
(generation). Small chunks split entities from facts and retrieval collapses; the reader can't save it.

Learning objectives:
1. Write chunking with overlap and state what small chunks split (entity ↔ evidence) and what overlap stitches.
2. Write TF-IDF + BM25 cosine retrieval from memory and state what idf rewards (rare = discriminative).
3. Report recall@k/MRR vs EM separately and say which half a miss belongs to.
4. Read the ablation table and argue chunk/overlap/top-k/rerank tradeoffs with numbers.
  Key lesson: generation quality is capped by retrieval quality — EM|miss ≈ 0 no matter the reader.


In [ ]:
import os # need os to make results/ and read cpu count
import random # need random to seed stdlib RNG
import re # need re for tokenize + sentence split + normalization
import time # need time to time indexing + retrieval
import csv # need csv to write retrieval.csv + eval.csv
import math # need math for log() in idf
import numpy as np # need numpy for TF-IDF/BM25 matrices + metrics
import matplotlib # need matplotlib for figures
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for recall/EM figure
SEED = 20 # fixed seed: same corpus + same splits every run
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
os.makedirs('./results', exist_ok=True) # create results/ for CSVs, PNG, prompt
print(f"seed={SEED} numpy={np.__version__}") # env line for the README


In [ ]:
# ---- text utils: tokenize + naive stem + normalize + sentences ----
def tok(s): # lowercase alphanumeric tokens with naive plural-strip stem
    ws = re.findall(r'[a-z0-9]+', s.lower()) # extract word tokens
    return [w[:-1] if (len(w) > 3 and w.endswith('s')) else w for w in ws] # strip trailing s (seats->seat)
def norm(s): # canonical form for EM: lowercase, drop non-alphanumerics
    return re.sub(r'[^a-z0-9]', '', s.lower()) # e.g. '34,000' -> '34000'
def has_num(sn): # standalone number token? (entity 'vorbren0' has digits but starts with a letter)
    return any(w[:1].isdigit() for w in tok(sn)) # true only for tokens like '34000', never entity names
assert has_num('It seats 34000 supporters') # fact sentences carry numbers
assert not has_num('The vorbren0 arena hosts matches') # entity digits do not count
def sents(doc): # split a doc into sentences on '. '
    return [x.strip() for x in doc.split('. ') if x.strip()] # drop empties
# ---- corpus: 6 topics x 8 docs; every doc owns a UNIQUE entity token; s2 holds the fact ----
SYL = ['vor','quen','hal','bren','zor','mil','tax','fen','gor','lax','den','rus'] # syllables for entity names
def entity(i): # unique single-token entity per doc (max idf by construction)
    return SYL[i % 12] + SYL[(i * 5 + 3) % 12] + str(i) # e.g. vorlax3 (unique via trailing id)
TOPICS = [ # (topic, context template with entity, FACT template without entity, question template)
    ('arena', 'The {e} arena in Norvost hosts winter matches', 'It seats {n} supporters on covered stands', 'How many supporters does the {e} arena seat'), # topic 1: stadium
    ('battery', 'The {e} pack ships in steel housings', 'It delivers {n} km of range on a full charge', 'How many km of range does the {e} pack deliver'), # topic 2: battery
    ('booster', 'The {e} booster stands on pad nine', 'It produces {n} kN of thrust at liftoff', 'How much thrust does the {e} booster produce in kN'), # topic 3: rocket
    ('bridge', 'The {e} bridge spans the Halden strait', 'It measures {n} meters from tower to tower', 'How many meters long is the {e} bridge'), # topic 4: bridge
    ('dam', 'The {e} dam holds the spring melt', 'It stores {n} million cubic meters at full pool', 'How many million cubic meters does the {e} dam store'), # topic 5: dam
    ('tunnel', 'The {e} tunnel runs under the old town', 'It extends {n} km underground through granite', 'How many km underground does the {e} tunnel extend')] # topic 6: tunnel
NUMS = ['34000', '410', '7900', '1250', '86', '17', '9200', '255'] # numeric answers cycle per doc
docs, meta = [], [] # doc texts + (topic, entity, answer) per doc
for i in range(48): # 48 docs: 6 topics x 8
    tp, s1t, factt, _ = TOPICS[i % 6] # topic cycles every 6
    e, n = entity(i), NUMS[i % 8] # unique entity + cycling number
    s1 = s1t.format(e=e) # context sentence (entity, no fact)
    s2 = factt.format(n=n) # FACT sentence (answer + keywords, NO entity: small chunks split them apart)
    s3 = f"Visitor to {e} praise the {tp} staff and the clear sign" # distractor: entity, no keywords
    s4 = f"The {tp} record book list every season since the rebuild" # distractor: keywords, no entity/answer
    f1 = "Guide mention the site on every tour" # filler: neutral words, pads entity-fact distance
    f2 = "Evening light favor the western gate" # filler: more distance (fact lands past word 40)
    docs.append('. '.join([s1, s3, f1, f2, s4, s2]) + '.') # 6-sentence doc (~51 words; fact last)
    meta.append((tp, e, n)) # stash gold answer + entity
print(f"docs={len(docs)} topics=6 entities_unique={len(set(m[1] for m in meta))}") # corpus card
assert len(set(m[1] for m in meta)) == 48 # every doc owns a unique entity token
# ---- questions: 24 (4 per topic); gold = doc id + answer span ----
Qs = [] # (qid, doc_id, question, gold_answer)
for t in range(6): # per topic
    ids = [i for i in range(48) if i % 6 == t][:4] # first 4 docs of this topic
    for di in ids: # one question per doc
        _, e, n = meta[di] # gold entity + answer
        q = TOPICS[t][3].format(e=e) + '?' # question reuses fact keywords + entity
        Qs.append((len(Qs), di, q, n)) # append with running qid
print(f"questions={len(Qs)} example: {Qs[0][2]} -> {Qs[0][3]}") # question card
# ---- chunker: word windows with overlap; track source doc + char span ----
def chunk_corpus(docs, size, overlap): # size/overlap in words
    chunks, src, spans = [], [], [] # text, doc id, (char_start, char_end)
    step = size - overlap # window stride
    assert step > 0 # overlap must be smaller than size
    for di, d in enumerate(docs): # per doc
        ws = d.split(' ') # word-split (keep raw words for display)
        if not ws: # guard empty docs
            continue # skip (never happens here)
        for s in range(0, len(ws), step): # sliding windows
            c = ' '.join(ws[s:s + size]) # window text
            chunks.append(c) # store
            src.append(di) # source doc
            spans.append((s, s + size)) # word span
            if s + size >= len(ws): # last window covers the tail
                break # avoid a duplicate tail window
    return chunks, np.array(src), spans # parallel lists
ct, st, _ = chunk_corpus(docs, 120, 0) # probe chunking at main config
assert len(ct) == 48 # 120-word windows cover each ~40-word doc in exactly one chunk
assert all(len(s.split(' ')) <= 120 for s in ct) # size contract honored
c30, s30, _ = chunk_corpus(docs, 30, 0) # probe small chunks
assert len(c30) > 48 # small chunks split docs into pieces (the ablation mechanism)
print(f"chunks@120={len(ct)} chunks@30={len(c30)} (small chunks split entity from evidence)") # chunk card


In [ ]:
# ---- embed from scratch: TF-IDF (log-tf x idf, L2 rows) + BM25 scorer ----
def build_vocab(chunk_toks): # token -> col over all chunks
    vb = {} # token -> index
    for ts in chunk_toks: # per chunk
        for w in ts: # per token
            if w not in vb: # first sight
                vb[w] = len(vb) # assign next column
    return vb # vocab map
def tfidf_index(chunk_toks, vb): # returns (X L2-row-normed, idf vec, doc lens not needed)
    N, J = len(chunk_toks), len(vb) # rows, cols
    df = np.zeros(J) # document frequency per term
    for ts in chunk_toks: # per chunk
        for w in set(ts): # once per chunk
            df[vb[w]] += 1 # count chunks containing w
    idf = np.log((N + 1) / (df + 1)) + 1.0 # smoothed idf: rare -> ~log N, common -> ~1
    X = np.zeros((N, J)) # dense TF-IDF (tiny corpus: fine)
    for i, ts in enumerate(chunk_toks): # per chunk
        for w in ts: # per token occurrence
            X[i, vb[w]] += 1.0 # raw tf
        nz = X[i] > 0 # nonzero cols
        X[i][nz] = (1.0 + np.log(X[i][nz])) * idf[nz] # log-tf x idf
        X[i] /= (np.linalg.norm(X[i]) + 1e-12) # L2 row norm -> cosine = dot
    return X, idf # index matrix + idf for queries
def tfidf_query(q, vb, idf): # same pipeline for one question
    v = np.zeros(len(vb)) # sparse-ish dense row
    for w in tok(q): # stemmed question tokens
        if w in vb: # ignore OOV
            v[vb[w]] += 1.0 # raw tf
    nz = v > 0 # nonzero cols
    v[nz] = (1.0 + np.log(v[nz])) * idf[nz] # log-tf x idf with chunk idf
    return v / (np.linalg.norm(v) + 1e-12) # L2 norm for cosine
def bm25_scores(q, chunk_toks, vb, df, avglen, k1=1.5, b=0.75): # Okapi BM25 per chunk
    N = len(chunk_toks) # chunk count
    idf = np.log((N - df + 0.5) / (df + 0.5) + 1.0) # BM25 idf (Robertson-Spärck Jones)
    sc = np.zeros(N) # scores
    qt = tok(q) # question tokens
    for i, ts in enumerate(chunk_toks): # per chunk
        L = len(ts) # chunk length
        for w in qt: # per query term
            if w not in vb: # OOV contributes nothing
                continue # skip
            j = vb[w] # column
            f = ts.count(w) # term freq in this chunk
            if f == 0: # absent term
                continue # skip
            sc[i] += idf[j] * f * (k1 + 1) / (f + k1 * (1 - b + b * L / avglen)) # saturated tf x length norm
    return sc # raw BM25 scores (not normalized: rank only)
# ---- retrieve + rerank + reader ----
def retrieve(q, X, vb, idf, topk): # cosine top-k chunk ids + scores
    s = X @ tfidf_query(q, vb, idf) # cosine vs every chunk (rows are L2-normed)
    o = np.argsort(-s)[:topk] # descending rank
    return o, s[o] # ids + scores
def rerank_bigram(q, cand, chunks): # bigram-overlap re-sort of candidate chunk ids
    qt = tok(q) # stemmed question tokens
    qb = set(zip(qt, qt[1:])) # question bigrams
    scored = [] # (overlap, chunk id)
    for c in cand: # per candidate
        ct = tok(chunks[c]) # chunk tokens
        cb = set(zip(ct, ct[1:])) # chunk bigrams
        scored.append((len(qb & cb), c)) # bigram overlap count
    scored.sort(reverse=True) # best overlap first (stable: ties keep TF-IDF order)
    return np.array([c for _, c in scored]) # re-ranked ids
def read_answer(q, chunk_ids, chunks): # extractive reader: best sentence by overlap + answer-type prior
    qq = set(tok(q)) # question token set
    quant = bool({'how', 'many', 'much'} & qq) # quantitative question? (all 24 here are)
    best, bs = '', -1 # best sentence + its score
    for c in chunk_ids: # across retrieved chunks in rank order
        for sn in sents(chunks[c]): # per sentence
            ov = len(qq & set(tok(sn))) # word overlap score
            sc = ov + (3 if (quant and has_num(sn)) else 0) # +3 if it carries a number (answer-type prior)
            if sc > bs: # strictly better replaces (rank order breaks ties)
                best, bs = sn, sc # keep it
    return best # top-scoring sentence (offline stand-in for the LLM)
# ---- unit tests: self-sim, OOV, BM25 probe, rerank stability ----
_ct = [tok(c) for c in ct] # tokenized main chunks
_vb = build_vocab(_ct) # vocab map
_X, _idf = tfidf_index(_ct, _vb) # main TF-IDF index
_self = _X @ _X[0] # cosine of every chunk vs chunk 0
assert abs(_self[0] - 1.0) < 1e-9 # self-similarity is exactly 1 (L2 rows)
assert np.all(_self[1:] < 0.999) # distinct docs are not identical
qv = tfidf_query('xyzzy unseenword', _vb, _idf) # all-OOV query
assert np.linalg.norm(qv) < 1e-9 # OOV query -> zero vector (safe, no crash)
_df = np.zeros(len(_vb)) # df vector for BM25 probe
for ts in _ct: # per chunk
    for w in set(ts): # once per chunk
        _df[_vb[w]] += 1 # df count
_al = float(np.mean([len(t) for t in _ct])) # average chunk length
_bs = bm25_scores(Qs[0][2], _ct, _vb, _df, _al) # BM25 for question 0
assert int(np.argmax(_bs)) == Qs[0][1] or st[int(np.argmax(_bs))] == Qs[0][1] # BM25 top-1 is the gold chunk/doc
_rr = rerank_bigram(Qs[0][2], np.arange(len(ct)), ct) # rerank over all chunks
assert set(_rr.tolist()) == set(range(len(ct))) # rerank is a permutation (never drops evidence)
print(f"vocab={len(_vb)} avglen={_al:.1f} unit-tests-pass") # index card


In [ ]:
# ---- metrics: recall@k + MRR over chunks; gold = chunk holding the answer span ----
def gold_chunk(qid, chunks, src, gold_doc, gold_ans): # chunk id containing the answer span in the gold doc
    ga = norm(gold_ans) # normalized answer
    for c in range(len(chunks)): # scan chunks in order
        if src[c] == gold_doc and ga in norm(chunks[c]): # gold doc + span present
            return c # first (and usually only) match
    return -1 # missing (should not happen: facts are planted verbatim)
def evaluate(chunks, src, scorer, topk, rerank=False): # scorer(q) -> ranked chunk ids
    r1 = r3 = r5 = mrr = 0.0 # accumulators
    hits = [] # per-question: gold chunk rank (0-based) or None
    for qid, gd, q, ga in Qs: # per question
        g = gold_chunk(qid, chunks, src, gd, ga) # gold chunk
        assert g >= 0 # every planted fact must be found verbatim
        rk = scorer(q) # full ranking of chunk ids
        if rerank: # rerank top-10 by bigram overlap, keep rest after
            rk = np.concatenate([rerank_bigram(q, rk[:10], chunks), rk[10:]]) # re-sorted head + tail
        pos = int(np.where(rk == g)[0][0]) # rank of gold chunk
        hits.append(pos) # store rank
        r1 += pos < 1 # recall@1
        r3 += pos < 3 # recall@3
        r5 += pos < 5 # recall@5
        mrr += 1.0 / (pos + 1) # reciprocal rank
    n = len(Qs) # 24 questions
    return {'r1': r1 / n, 'r3': r3 / n, 'r5': r5 / n, 'mrr': mrr / n, 'k': topk} # metric dict
def scorer_for(chunks, kind): # build a scorer closure for TF-IDF or BM25
    ctk = [tok(c) for c in chunks] # tokenize once
    if kind == 'tfidf': # TF-IDF cosine branch
        vb = build_vocab(ctk) # vocab
        X, idf = tfidf_index(ctk, vb) # index
        return lambda q: np.argsort(-(X @ tfidf_query(q, vb, idf))) # ranked ids
    vb = build_vocab(ctk) # BM25 shares vocab builder
    df = np.zeros(len(vb)) # df vector
    for ts in ctk: # per chunk
        for w in set(ts): # once per chunk
            df[vb[w]] += 1 # df count
    al = float(np.mean([len(t) for t in ctk])) # avg length
    return lambda q: np.argsort(-bm25_scores(q, ctk, vb, df, al)) # ranked ids
t0 = time.time() # wall clock for the eval block
main_chunks, main_src, _ = chunk_corpus(docs, 120, 0) # main config: ~1 chunk/doc
sc_tfidf = scorer_for(main_chunks, 'tfidf') # TF-IDF scorer
sc_bm25 = scorer_for(main_chunks, 'bm25') # BM25 scorer
m_t = evaluate(main_chunks, main_src, sc_tfidf, 3, False) # TF-IDF metrics
m_b = evaluate(main_chunks, main_src, sc_bm25, 3, False) # BM25 metrics
m_tr = evaluate(main_chunks, main_src, sc_tfidf, 3, True) # TF-IDF + rerank metrics
print(f"tfidf r@1={m_t['r1']:.3f} r@3={m_t['r3']:.3f} r@5={m_t['r5']:.3f} mrr={m_t['mrr']:.3f}") # TF-IDF card
print(f"bm25  r@1={m_b['r1']:.3f} r@3={m_b['r3']:.3f} r@5={m_b['r5']:.3f} mrr={m_b['mrr']:.3f}") # BM25 card
print(f"+rerank r@3={m_tr['r3']:.3f} mrr={m_tr['mrr']:.3f} ({time.time()-t0:.1f}s)") # rerank card
assert m_t['r1'] <= m_t['r3'] <= m_t['r5'] # recall monotone in k by construction
assert m_t['r3'] >= 0.90 and m_t['mrr'] >= 0.80 # main config must be near-perfect (unique entities)
assert m_tr['mrr'] >= m_t['mrr'] - 0.01 # rerank must not wreck a good ranking (ties expected here)


In [ ]:
# ---- ablations: chunk size / overlap / top-k / rerank; generation split; save + figure ----
rows = [] # retrieval.csv rows
for size, ov in [(30, 0), (30, 10), (60, 0), (120, 0)]: # chunk/overlap sweep
    ch, sr, _ = chunk_corpus(docs, size, ov) # chunk at this config
    sc = scorer_for(ch, 'tfidf') # TF-IDF scorer
    for rr in [False, True]: # rerank off/on
        m = evaluate(ch, sr, sc, 3, rr) # metrics (recall@1/3/5 + MRR)
        rows.append({'chunk': size, 'overlap': ov, 'retriever': 'tfidf', 'rerank': int(rr), # config cols
                     'recall@1': round(m['r1'], 4), 'recall@3': round(m['r3'], 4), # metric cols
                     'recall@5': round(m['r5'], 4), 'mrr': round(m['mrr'], 4)}) # metric cols
        print(rows[-1]) # one line per config
r30 = [r for r in rows if r['chunk'] == 30 and r['rerank'] == 0][0] # small-chunk row
r120 = [r for r in rows if r['chunk'] == 120 and r['rerank'] == 0][0] # whole-doc row
assert r120['recall@3'] >= r30['recall@3'] # whole docs beat split evidence (the chunking lesson)
assert r120['mrr'] >= 0.80 # main config sanity repeated on the sweep path
with open('./results/retrieval.csv', 'w', newline='') as f: # save sweep table
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(rows) # one row per config
print('saved ./results/retrieval.csv') # confirm artifact
# ---- generation: extractive reader @ main chunks vs @ chunk-30 vs closed-book ----
def generation_em(chunks, src, k): # EM over 24 questions reading top-k chunks
    sc = scorer_for(chunks, 'tfidf') # fresh scorer for these chunks
    ems, erows = [], [] # per-question EM + csv rows
    for qid, gd, q, ga in Qs: # per question
        rk = sc(q)[:k] # top-k chunks
        pred = read_answer(q, rk, chunks) # best-overlap sentence
        em = int(norm(ga) in norm(pred)) # EM = gold span contained in predicted sentence
        g = gold_chunk(qid, chunks, src, gd, ga) # gold chunk
        hit = int(g in rk) # retrieved?
        ems.append(em) # store
        erows.append({'qid': qid, 'chunk_n': len(chunks), 'hit': hit, 'em': em, # eval cols
                      'pred': pred[:90]}) # truncated prediction
    return float(np.mean(ems)), erows # mean EM + rows
em_main, rows_main = generation_em(main_chunks, main_src, 3) # reader on whole-doc chunks
ch30, sr30, _ = chunk_corpus(docs, 30, 0) # small-chunk corpus
em_c30, rows_c30 = generation_em(ch30, sr30, 3) # reader on split-evidence chunks
em_closed = 0.0 # closed-book: no context -> empty prediction -> EM 0 by construction
hit_main = float(np.mean([r['hit'] for r in rows_main])) # hit@3 rate at main config
h30 = [r['em'] for r in rows_c30 if r['hit'] == 1] # EMs where evidence was retrieved (chunk-30)
m30 = [r['em'] for r in rows_c30 if r['hit'] == 0] # EMs where evidence was missed (chunk-30)
em_hit = float(np.mean(h30)) if h30 else 1.0 # EM|hit (empty -> vacuous 1.0, never asserts alone)
em_miss = float(np.mean(m30)) if m30 else 0.0 # EM|miss (empty -> vacuous 0.0)
print(f"EM main={em_main:.3f} (hit@3={hit_main:.3f}) EM chunk30={em_c30:.3f} closed-book={em_closed:.3f}") # EM card
print(f"split@chunk30: n_hit={len(h30)} EM|hit={em_hit:.3f} n_miss={len(m30)} EM|miss={em_miss:.3f}") # split card
assert em_main >= 0.60 # reader must extract well when evidence is retrieved
assert em_main - em_closed > 0.40 # RAG beats closed-book by a wide margin
assert em_main + 1e-9 >= em_c30 # better retrieval must not hurt generation
assert em_hit + 1e-9 >= em_miss # retrieved evidence must not hurt extraction (misses guess by number collision)
with open('./results/eval.csv', 'w', newline='') as f: # save per-question eval
    w = csv.DictWriter(f, fieldnames=list(rows_main[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(rows_main) # 24 question rows
print('saved ./results/eval.csv') # confirm artifact
q0 = Qs[0] # first question for the prompt example
rk0 = sc_tfidf(q0[2])[:3] # its top-3 chunks
with open('./results/prompt_example.txt', 'w') as f: # save the prompt we would send the LLM
    f.write('Context:\n') # context header
    for c in rk0: # per retrieved chunk
        f.write(f"- {main_chunks[c]}\n") # bullet chunk
    f.write(f"\nQuestion: {q0[2]}\nAnswer:") # question + answer cue
print('saved ./results/prompt_example.txt') # confirm artifact
fig, ax = plt.subplots(1, 3, figsize=(15, 4)) # three panels: retriever, chunks, EM split
ax[0].bar(['tfidf r@1', 'tfidf r@3', 'bm25 r@1', 'bm25 r@3'], [m_t['r1'], m_t['r3'], m_b['r1'], m_b['r3']]) # recall bars
ax[0].set_ylim(0, 1.05) # recall range
ax[0].set_ylabel('recall') # y label
ax[0].set_title('P20: TF-IDF vs BM25') # panel title
cfg = [f"{r['chunk']}/{r['overlap']}" for r in rows if r['rerank'] == 0] # chunk/overlap labels
ax[1].bar(cfg, [r['recall@3'] for r in rows if r['rerank'] == 0], label='recall@3') # recall@3 per config
ax[1].bar(cfg, [r['mrr'] for r in rows if r['rerank'] == 0], alpha=0.5, label='mrr') # MRR per config
ax[1].tick_params(axis='x', rotation=15) # tilt labels
ax[1].set_ylim(0, 1.05) # metric range
ax[1].set_title('P20: chunk/overlap sweep') # panel title
ax[1].legend() # show series
ax[2].bar(['closed-book', 'RAG c30', 'RAG main', 'EM|hit', 'EM|miss'], [em_closed, em_c30, em_main, em_hit, em_miss]) # EM bars
ax[2].set_ylim(0, 1.05) # EM range
ax[2].set_ylabel('EM') # y label
ax[2].set_title('P20: retrieval caps generation') # panel title
fig.tight_layout() # avoid overlap
fig.savefig('./results/rag.png', dpi=100) # save figure
print('saved ./results/rag.png') # confirm artifact


Cleanup / next steps:
- Artifacts: `results/retrieval.csv` (chunk/overlap/rerank sweep), `results/eval.csv` (24 per-question
  hit/EM rows), `results/prompt_example.txt` (the prompt an LLM would get), `results/rag.png`.
- Rerun anywhere: corpus is synthetic + seeded, zero downloads, numpy-only (CPU seconds).
- Next: P21 wraps a tool loop around an LM (calculator / fs / python / db / http + retries, timeouts,
  permissions) — retrieval becomes one tool among many, and every call is metered.
